# VGGT M0 CUDA reference

Runs the approved four-view fixture twice on a Colab CUDA GPU, compares determinism, and downloads the compact JSON evidence bundle. Select **Runtime → Change runtime type → T4 GPU** (or better) before running all cells. The original `facebook/VGGT-1B` checkpoint is CC BY-NC 4.0 and is approved here only for this strictly non-commercial project.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing'
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone https://github.com/koernergb/vggt-in-browser.git
%cd vggt-in-browser
!git log -1 --oneline

In [ ]:
# Use the upstream-pinned CUDA versions and exact VGGT source revision.
%pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r model/reference/requirements.txt
%pip install -q 'vggt @ git+https://github.com/facebookresearch/vggt.git@a288dd0f14786c93483e45524328726ab7b1b4ce'

The next cell downloads four small fixture images and the approximately 5.03 GB non-commercial checkpoint. Colab may take several minutes.

In [ ]:
!python model/reference/fetch_fixture.py
!python model/reference/validate_fixture.py bench/fixtures/manifest.json

In [ ]:
!python model/reference/run_reference.py --fixture bench/fixtures/manifest.json --device cuda --dtype auto --max-views 4 --output bench/results/m0/cuda-4view-run-01.json
!python model/reference/run_reference.py --fixture bench/fixtures/manifest.json --device cuda --dtype auto --max-views 4 --output bench/results/m0/cuda-4view-run-02.json
!python model/reference/compare_runs.py bench/results/m0/cuda-4view-run-01.json bench/results/m0/cuda-4view-run-02.json --output bench/results/m0/cuda-4view-determinism.json

In [ ]:
import json, shutil
from pathlib import Path
from google.colab import files
result_dir = Path('bench/results/m0')
for path in sorted(result_dir.glob('cuda-*.json')):
    data = json.loads(path.read_text())
    assert data.get('classification') == 'measured'
    print(path.name, data.get('timings_seconds', data.get('exactly_deterministic')))
archive = shutil.make_archive('vggt-m0-cuda-results', 'zip', result_dir)
files.download(archive)